# 1 EDA - Dataset limpo UFC

Objetivo: olhar o dataset final salvo em `ufc-master_more_clean.csv`, entender quantos dados sobraram e listar quais features estao disponiveis para a proxima etapa de EDA/modelagem.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 120)

csv_path = Path('ufc-master_more_clean.csv')
df = pd.read_csv(csv_path, parse_dates=['date'])

print(f'Dataset: {csv_path.name}')
print(f'Linhas: {df.shape[0]:,}')
print(f'Colunas: {df.shape[1]:,}')

df.head()

Dataset: ufc-master_more_clean.csv
Linhas: 2,625
Colunas: 45


,date,R_fighter,B_fighter,Winner,weight_class,gender,no_of_rounds,R_current_win_streak,B_current_win_streak,R_longest_win_streak,B_longest_win_streak,R_current_lose_streak,B_current_lose_streak,R_wins,B_wins,R_losses,B_losses,R_total_rounds_fought,B_total_rounds_fought,R_total_title_bouts,B_total_title_bouts,R_win_by_KO/TKO,B_win_by_KO/TKO,R_win_by_Submission,B_win_by_Submission,R_Height_cms,B_Height_cms,R_Reach_cms,B_Reach_cms,R_age,B_age,R_avg_SIG_STR_landed,B_avg_SIG_STR_landed,R_avg_SIG_STR_pct,B_avg_SIG_STR_pct,R_avg_SUB_ATT,B_avg_SUB_ATT,R_avg_TD_landed,B_avg_TD_landed,R_avg_TD_pct,B_avg_TD_pct,R_Stance,B_Stance,R_ufc_fights,B_ufc_fights
0,2026-03-28,Israel Adesanya,Joe Pyfer,Blue,Middleweight,MALE,5,0,3,9,4,3,0,13,7,5,2,66,18,12,0,5,4,0,2,193.04,187.96,203.20,190.50,36,29,4.03,3.52,0.48,0.44,0.1,0.9,0.05,1.45,0.09,0.30,Switch,Orthodox,18,9
1,2026-03-28,Michael Chiesa,Niko Price,Red,Welterweight,MALE,3,3,0,4,2,0,3,14,8,7,10,47,40,1,0,0,4,8,2,185.42,182.88,190.50,193.04,38,36,2.02,5.11,0.40,0.43,1.0,0.6,3.11,1.06,0.47,0.30,Southpaw,Orthodox,21,18
2,2026-03-28,Mansur Abdul-Malik,Yousri Belgaroui,Blue,Middleweight,MALE,3,4,2,3,2,0,0,4,2,0,1,9,9,0,0,3,2,1,0,187.96,198.12,203.20,200.66,28,33,3.28,6.10,0.44,0.64,0.3,0.0,1.65,0.29,0.41,1.00,Orthodox,Orthodox,4,3
3,2026-03-28,Terrance McKinney,Kyle Nelson,Red,Lightweight,MALE,3,0,1,2,3,1,0,7,5,6,5,16,26,0,0,4,2,3,0,177.80,180.34,185.42,180.34,31,34,6.53,3.60,0.55,0.45,2.1,0.5,3.31,1.18,0.40,0.23,Switch,Switch,13,10
4,2026-03-28,Navajo Stirling,Bruno Lopes,Red,Light Heavyweight,MALE,3,4,0,4,2,0,1,4,2,0,2,11,7,0,0,1,1,0,0,193.04,187.96,200.66,187.96,28,32,6.25,2.88,0.52,0.43,0.0,0.0,0.98,1.93,0.28,0.21,Orthodox,Orthodox,4,4


## 1.1 O que sobrou no dataset?

Esse CSV representa o recorte mais limpo ate agora: lutas masculinas, categorias escolhidas, periodo moderno e ambos os lutadores com pelo menos 2 lutas previas no UFC.

In [ ]:
resumo_dataset = pd.DataFrame({
    'metrica': [
        'linhas',
        'colunas',
        'data_inicial',
        'data_final',
        'categorias_de_peso',
        'lutadores_unicos',
    ],
    'valor': [
        len(df),
        df.shape[1],
        df['date'].min().date(),
        df['date'].max().date(),
        df['weight_class'].nunique(),
        pd.concat([df['R_fighter'], df['B_fighter']]).nunique(),
    ]
})

resumo_dataset

,metrica,valor
0,linhas,2625
1,colunas,45
2,data_inicial,2015-01-03
3,data_final,2026-03-28
4,categorias_de_peso,6
5,lutadores_unicos,1019


## 1.2 Features disponiveis

Abaixo esta a lista completa das colunas que sobraram, com indice para facilitar referencia durante a EDA.

In [ ]:
features = pd.DataFrame({
    'idx': range(len(df.columns)),
    'feature': df.columns,
    'tipo': df.dtypes.astype(str).values,
    'qtd_faltante': df.isna().sum().values,
    'pct_faltante': (df.isna().mean().values * 100).round(2),
})

features

,idx,feature,tipo,qtd_faltante,pct_faltante
0,0,date,datetime64[us],0,0.00
1,1,R_fighter,str,0,0.00
2,2,B_fighter,str,0,0.00
3,3,Winner,str,0,0.00
4,4,weight_class,str,0,0.00
5,5,gender,str,0,0.00
6,6,no_of_rounds,int64,0,0.00
7,7,R_current_win_streak,int64,0,0.00
8,8,B_current_win_streak,int64,0,0.00
9,9,R_longest_win_streak,int64,0,0.00


## 1.3 Features por grupo

Separando as colunas por papel ajuda a decidir o que pode entrar como variavel explicativa e o que deve ficar fora do modelo.

In [ ]:
colunas_identificacao = ['date', 'R_fighter', 'B_fighter']
target = ['Winner']
colunas_contexto = ['weight_class', 'gender', 'no_of_rounds']
colunas_blue = [col for col in df.columns if col.startswith('B_')]
colunas_red = [col for col in df.columns if col.startswith('R_')]
colunas_experiencia = ['R_ufc_fights', 'B_ufc_fights']

grupos_features = {
    'identificacao': colunas_identificacao,
    'target': target,
    'contexto_da_luta': colunas_contexto,
    'blue_corner': colunas_blue,
    'red_corner': colunas_red,
    'experiencia_ufc': colunas_experiencia,
}

for grupo, colunas in grupos_features.items():
    print(f'\n{grupo} ({len(colunas)} colunas)')
    display(pd.DataFrame({'feature': colunas}))


identificacao (3 colunas)


,feature
0,date
1,R_fighter
2,B_fighter



target (1 colunas)


,feature
0,Winner



contexto_da_luta (3 colunas)


...

,feature
0,B_fighter
1,B_current_win_streak
2,B_longest_win_streak
3,B_current_lose_streak
4,B_wins
5,B_losses
6,B_total_rounds_fought
7,B_total_title_bouts
8,B_win_by_KO/TKO
9,B_win_by_Submission



red_corner (20 colunas)


,feature
0,R_fighter
1,R_current_win_streak
2,R_longest_win_streak
3,R_current_lose_streak
4,R_wins
5,R_losses
6,R_total_rounds_fought
7,R_total_title_bouts
8,R_win_by_KO/TKO
9,R_win_by_Submission



experiencia_ufc (2 colunas)


,feature
0,R_ufc_fights
1,B_ufc_fights


## 1.4 Recortes rapidos

Conferencias basicas para entender a distribuicao do dataset final.

In [ ]:
recortes = {
    'vencedor_red_blue': df['Winner'].value_counts(dropna=False),
    'genero': df['gender'].value_counts(dropna=False),
    'categorias': df['weight_class'].value_counts(dropna=False),
    'rounds_previstos': df['no_of_rounds'].value_counts(dropna=False).sort_index(),
    'stance_red': df['R_Stance'].value_counts(dropna=False),
    'stance_blue': df['B_Stance'].value_counts(dropna=False),
    'years': df['date'].dt.year.value_counts(dropna=False).sort_index(),
}

for nome, serie in recortes.items():
    print(f'\n{nome}')
    display(serie)


vencedor_red_blue


Winner
Red           1467
Blue          1152
Draw             4
No Contest       2
Name: count, dtype: int64


genero


gender
MALE    2625
Name: count, dtype: int64


categorias


...

R_Stance
Orthodox    1837
Southpaw     560
Switch       228
Name: count, dtype: int64


stance_blue


B_Stance
Orthodox    1802
Southpaw     588
Switch       235
Name: count, dtype: int64


years


date
2015    221
2016    242
2017    201
2018    219
2019    210
2020    175
2021    247
2022    255
2023    242
2024    275
2025    281
2026     57
Name: count, dtype: int64

## 1.5 Faltantes

Aqui vemos se ainda existem buracos depois dos filtros.

In [ ]:
faltantes = (
    df.isna().sum()
    .sort_values(ascending=False)
    .rename('qtd_faltante')
    .to_frame()
)
faltantes['pct_faltante'] = (faltantes['qtd_faltante'] / len(df) * 100).round(2)

faltantes.query('qtd_faltante > 0')

,qtd_faltante,pct_faltante
R_avg_SIG_STR_landed,42,1.60
B_avg_SIG_STR_landed,42,1.60
B_avg_TD_pct,3,0.11
R_avg_TD_pct,3,0.11


## 1.6 Investigação da confiabilidade usando histórico de lutas de um atleta

In [ ]:
adesanya_fights = df[
    (df['R_fighter'] == 'Israel Adesanya') |
    (df['B_fighter'] == 'Israel Adesanya')
]


charles_oliveira_fights = df[
    (df['R_fighter'] == 'Charles Oliveira') |   
    (df['B_fighter'] == 'Charles Oliveira')
]   



charles_oliveira_fights


,date,R_fighter,B_fighter,Winner,weight_class,gender,no_of_rounds,R_current_win_streak,B_current_win_streak,R_longest_win_streak,B_longest_win_streak,R_current_lose_streak,B_current_lose_streak,R_wins,B_wins,R_losses,B_losses,R_total_rounds_fought,B_total_rounds_fought,R_total_title_bouts,B_total_title_bouts,R_win_by_KO/TKO,B_win_by_KO/TKO,R_win_by_Submission,B_win_by_Submission,R_Height_cms,B_Height_cms,R_Reach_cms,B_Reach_cms,R_age,B_age,R_avg_SIG_STR_landed,B_avg_SIG_STR_landed,R_avg_SIG_STR_pct,B_avg_SIG_STR_pct,R_avg_SUB_ATT,B_avg_SUB_ATT,R_avg_TD_landed,B_avg_TD_landed,R_avg_TD_pct,B_avg_TD_pct,R_Stance,B_Stance,R_ufc_fights,B_ufc_fights
16,2026-03-07,Max Holloway,Charles Oliveira,Blue,Lightweight,MALE,5,1,1,13,11,0,0,23,24,8,11,106,71,10,4,11,4,2,17,180.34,177.80,175.26,187.96,34,36,6.9100,3.2300,0.480,0.550,0.3000,2.6000,0.2300,2.2900,0.530,0.390,Orthodox,Orthodox,31,35
111,2025-10-11,Charles Oliveira,Mateusz Gamrot,Red,Lightweight,MALE,5,0,1,11,4,1,0,23,8,11,3,69,30,4,0,4,3,16,1,177.80,177.80,187.96,177.80,35,34,3.2300,3.2900,0.550,0.510,2.6000,0.1000,2.2900,5.1500,0.390,0.370,Orthodox,Southpaw,34,11
199,2025-06-28,Ilia Topuria,Charles Oliveira,Red,Lightweight,MALE,5,8,1,8,11,0,0,8,23,0,10,19,68,2,3,5,4,1,16,170.18,177.80,175.26,187.96,28,35,4.8100,3.2300,0.480,0.550,1.1000,2.6000,1.9600,2.2900,0.610,0.390,Orthodox,Orthodox,8,33
360,2024-11-16,Charles Oliveira,Michael Chandler,Red,Lightweight,MALE,3,0,0,11,3,1,1,22,4,10,3,63,14,3,1,4,3,16,1,177.80,172.72,187.96,180.34,35,38,3.4000,4.4900,0.550,0.500,2.7000,0.6000,2.2500,1.5900,0.400,0.370,Orthodox,Orthodox,32,7
534,2024-04-13,Charles Oliveira,Arman Tsarukyan,Blue,Lightweight,MALE,3,1,3,11,5,0,0,22,8,9,2,60,27,3,0,4,4,16,0,177.80,170.18,187.96,182.88,34,27,3.3900,3.7900,0.530,0.480,2.8000,0.0000,2.1700,3.2500,0.400,0.370,Orthodox,Orthodox,31,10
757,2023-06-10,Charles Oliveira,Beneil Dariush,Red,Lightweight,MALE,3,0,8,11,8,1,0,21,16,9,4,59,45,3,0,3,3,16,5,177.80,177.80,187.96,182.88,33,34,3.3900,3.7800,0.530,0.480,2.8000,0.9000,2.1700,1.9000,0.400,0.340,Orthodox,Southpaw,30,20
905,2022-10-22,Charles Oliveira,Islam Makhachev,Blue,Lightweight,MALE,5,11,10,11,10,0,0,21,11,8,1,57,26,2,3,3,2,16,5,177.80,177.80,187.96,177.80,33,30,3.3900,2.6500,0.530,0.590,2.8000,1.1000,2.1700,3.1900,0.400,0.530,Orthodox,Southpaw,29,12
1036,2022-05-07,Charles Oliveira,Justin Gaethje,Red,Lightweight,MALE,5,10,1,10,4,0,0,20,6,8,3,56,22,3,2,3,5,15,0,177.80,180.34,187.96,177.80,32,33,3.3900,6.7800,0.530,0.590,2.8000,0.0000,2.1700,0.1100,0.400,0.250,Orthodox,Orthodox,28,9
1117,2021-12-11,Charles Oliveira,Dustin Poirier,Red,Lightweight,MALE,5,9,3,9,5,0,0,19,21,8,6,53,65,2,3,3,11,14,3,177.80,175.26,187.96,182.88,32,32,3.3900,5.3000,0.530,0.500,2.8000,1.3000,2.1700,1.2400,0.400,0.360,Orthodox,Southpaw,27,27
1266,2021-05-15,Charles Oliveira,Michael Chandler,Red,Lightweight,MALE,5,8,3,8,3,0,0,18,3,8,0,51,4,0,0,2,2,14,1,177.80,172.72,187.96,180.34,31,35,3.2600,4.4900,0.520,0.500,2.7000,1.6000,2.6400,2.0300,0.440,0.410,Orthodox,Orthodox,26,3


# 2 Feature Engineer